In [1]:
import os
import cv2
import pandas as pd

In [2]:
# current path is fetched. It is the parent directory of the zipped VOC dataset folder.
# this directory also contains data split -- text files

current_path = os.getcwd()


In [3]:
## specifying classes
filter = ['aeroplane','bicycle','bird','boat','bottle',
           'bus','car','cat','chair','cow',
           'diningtable','dog','horse','motorbike','person',
           'pottedplant','sheep','sofa','train','tvmonitor']

df_cats = pd.DataFrame(list(filter), 
               columns =['bbx_0_name'])

In [4]:
df_cats['bbx_0_name'] = df_cats['bbx_0_name'].astype('category')
df_cats['Categories'] = df_cats['bbx_0_name'].cat.codes+1
df_cats

,bbx_0_name,Categories
0,aeroplane,1
1,bicycle,2
2,bird,3
3,boat,4
4,bottle,5
5,bus,6
6,car,7
7,cat,8
8,chair,9
9,cow,10


In [5]:
#### getting image names from train.txt, val.txt and test.txt for classification task #####
### appending image info to np arrays

image_dir = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009')

image_list = open('class_train.txt')
imagenames_train = list()
images_train = list()
for line in image_list:
    imagenames_train.append(line.rstrip())    
    file = os.path.join(image_dir, 'JPEGImages', line.strip()+'.jpg')
    img = cv2.imread(file)
    img = cv2.resize(img,(224,224))
    images_train.append(img) 
df_train = pd.DataFrame(list(zip(imagenames_train, images_train)), 
               columns =['fileID', 'Image'])    

image_list = open('class_val.txt')
imagenames_val = list()
images_val = list()
for line in image_list:
    imagenames_val.append(line.rstrip())    
    file = os.path.join(image_dir, 'JPEGImages', line.strip()+'.jpg')
    img = cv2.imread(file)
    img = cv2.resize(img,(224,224))
    images_val.append(img) 
df_val = pd.DataFrame(list(zip(imagenames_val, images_val)), 
               columns =['fileID', 'Image']) 

image_list = open('class_test.txt')
imagenames_test = list()
images_test = list()
for line in image_list:
    imagenames_test.append(line.rstrip())
    file = os.path.join(image_dir, 'JPEGImages', line.strip()+'.jpg')
    img = cv2.imread(file)
    img = cv2.resize(img,(224,224))
    images_test.append(img) 
df_test = pd.DataFrame(list(zip(imagenames_test, images_test)), 
               columns =['fileID', 'Image'])     

In [ ]:
## specifying directories for Annotation xmls and JPEG images
dir_anno = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','Annotations')
img_dir  = os.path.join(current_path,'VOCtrainval_11-May-2009','VOCdevkit','VOC2009','JPEGImages')

In [ ]:
### extracting all relevant details by parsing annotation xmls
### unwanted information commented out 

import os 
import numpy as np
import xml.etree.ElementTree as ET
from collections import OrderedDict
import matplotlib.pyplot as plt
import pandas as pd 

def extract_single_xml_file(tree):
    Nobj = 0
    row  = OrderedDict()
    for elems in tree.iter():

#         if elems.tag == "size":
#             for elem in elems:
#                 row[elem.tag] = int(elem.text)
        if elems.tag == "object":
            for elem in elems:
                if elem.tag == "name":
                    row["bbx_{}_{}".format(Nobj,elem.tag)] = str(elem.text)              
#                 if elem.tag == "bndbox":
#                     for k in elem:
#                         row["bbx_{}_{}".format(Nobj,k.tag)] = float(k.text)
                    Nobj += 1
    row["Nobj"] = Nobj
    return(row)

df_anno = []
for fnm in os.listdir(dir_anno):  
    if not fnm.startswith('.'): ## do not include hidden folders/files
        tree = ET.parse(os.path.join(dir_anno,fnm))
        row = extract_single_xml_file(tree)
        row["fileID"] = fnm.split(".")[0]
        df_anno.append(row)
df_anno = pd.DataFrame(df_anno)

maxNobj = np.max(df_anno["Nobj"])


print("columns in df_anno\n-----------------")
for icol, colnm in enumerate(df_anno.columns):
    print("{:3.0f}: {}".format(icol,colnm))
print("-"*30)
print("df_anno.shape={}=(N frames, N columns)".format(df_anno.shape))
df_anno.head()

In [8]:
## STEP 1 in generating train, test and validation dataframes 
## 1. get bbx_0_names frpom annotation datasets for file ids in each set
## 2. keep relevant columns

df_annot_train = df_anno[df_anno['fileID'].isin(imagenames_train)]
df_annot_train = df_annot_train[['fileID','bbx_0_name']]
df_annot_val = df_anno[df_anno['fileID'].isin(imagenames_val)]
df_annot_val = df_annot_val[['fileID','bbx_0_name']]
df_annot_test = df_anno[df_anno['fileID'].isin(imagenames_test)]
df_annot_test = df_annot_test[['fileID','bbx_0_name']]

In [9]:
## STEP 2: merge categories into train, test and val dfs

df_annot_train = pd.merge(df_annot_train, df_cats, how='left', on=['bbx_0_name'])
df_annot_val = pd.merge(df_annot_val, df_cats, how='left', on=['bbx_0_name'])
df_annot_test = pd.merge(df_annot_test, df_cats, how='left', on=['bbx_0_name'])

In [10]:
## STEP 3: Get image matrices from the data reading operation done earlier
## merge images into train, test and val dfs

df_train = pd.merge(df_annot_train, df_train, how='inner', on=['fileID'])
df_val = pd.merge(df_annot_val, df_val, how='inner', on=['fileID'])
df_test = pd.merge(df_annot_test, df_test, how='inner', on=['fileID'])

In [15]:
## installing the correct numpy version (there were errors while loading npy files)
!pip install numpy==1.16.1
import numpy as np

     |████████████████████████████████| 17.3 MB 77 kB/s  eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 1.18.1
    Uninstalling numpy-1.18.1:
      Successfully uninstalled numpy-1.18.1


In [14]:
np.save('./x_val.npy', x_val)
np.save('./y_val.npy', y_val)
np.save('./x_train.npy', x_train)
np.save('./y_train.npy', y_train)
np.save('./x_test.npy', x_test)
np.save('./y_test.npy', y_test)

In [ ]:
##################################################
##################################################
##################################################
AFTER THIS POINT WORKING IN COLAB. USING COLAB GPU.

In [13]:
x_train = df_train['Image']
x_val =  df_val['Image']
x_test = df_test['Image']

y_train = df_train['Categories']
y_val = df_val['Categories']
y_test = df_test['Categories']

In [ ]:
## reshaping the np arrays (not for labels) to required dimensions (num images, img_size, img_size, classes)
x_train = np.array([x for x in x_train])
x_val = np.array([x for x in x_val])
x_test = np.array([x for x in x_test])


In [ ]:
## restructuring labels (one-hot encoding)
from keras.utils import to_categorical
y_train = to_categorical(y_train)
y_val = to_categorical(y_val)
# y_test = to_categorical(y_test)


In [ ]:
## import pretrained resnet50 without top layers

from keras.preprocessing.image import load_img
from keras.preprocessing.image import img_to_array
from keras.applications.vgg16 import preprocess_input
from keras.applications.vgg16 import decode_predictions
from keras.applications.vgg16 import VGG16
from keras.layers import Flatten, Dense
from keras.models import Model
import tensorflow as tf
# print(tf.__version__)

In [ ]:
# !pip install tensorflow_gpu==2.0

In [ ]:
## using vgg16 pretrained model

model = VGG16(include_top= False, input_shape=(224,224,3))

# # specifying additional layers
# # The dense layer: Flatten() accepts the output from the last layer of the base model as an input to the new model 
x = Flatten()(model.layers[-1].output)
# # next, a dense layer with softmax activation is connected. number of classes is set as 2
x = Dense(4096, activation='relu')(x)
x = Dense(21, activation = 'softmax',name='classification')(x)

# # full model is now specified alongwith the pre-specified input and expected output 
model = Model(inputs = model.input, outputs = x)
model.summary()

# # compiling model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])


for layer in model.layers[:-3]:
    layer.trainable = False

In [ ]:
model.fit(x_train,y_train, epochs=10, validation_data = (x_val,y_val),batch_size= 64)
model.save('Object_classification_30042020_10epochs_RV.h5')

In [ ]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(x_test)
y_pred =np.argmax(y_pred, axis=1)
print("Accuracy: {0:0.1f}%".format(accuracy_score(y_test,y_pred)*100))